# 실제 언더필 사진 50장과 300장으로 GAN 학습 비교

## 목적과 입력
언더필 출력물 표면에서 베드가 거의 보이지 않는 사진을 선별해 50장과 300장으로 각각 GAN을 처음부터 학습합니다. 50장은 300장 안에 포함되며, 두 경우 모두 사진 50장을 생성합니다. 사진 자동 선별은 완벽한 베드 제거 또는 모든 사진의 결함 가시성을 보장하지 않습니다.

사진은 기존 시간 블록 분리에서 지정된 학습 구간을 활용합니다. 기존 분류기 평가·검증·경계 버퍼 구간과 제외 규칙은 함께 포함된 input_manifest.csv / selection_settings.json에서 확인하세요. 연속 촬영에 따라 학습 사진이 서로 비슷할 수 있고 원본은 클래스당 출력 1회이므로 새 출력에 대한 일반화는 확인하지 못합니다.

## 같은 조건으로 비교하는 이유
기존 실험의 DCGAN을 **독립적으로 처음부터** 학습하며, 각 조건에서 입력은 동일한 고정 1000×1000 크롭을 256×256으로 축소한 RGB 이미지, 출력은 256×256 이미지입니다. 두 모델 모두 시드 42, 배치 8, 업데이트 1,500회, 같은 난수 시드로 출력 50장을 만듭니다. 모델 수량 가드만 일반화한 복사본을 쓰고 노트북이 실제 장수와 포함 관계를 검증합니다.

이 설정은 모델 구조·업데이트 예산을 맞추기 위한 예비 조건이며 최적화된 설정이 아닙니다. 무작위 복원 추출이어서 학습 사진이 선택되는 평균 횟수는 50장 조건에서 약 240회, 300장 조건에서 약 40회입니다. 따라서 결과 차이를 데이터 수 하나의 순수한 효과라고 단정할 수 없습니다. 더 많은 업데이트가 필요한지는 별도 실험이 필요합니다.

이번 작업은 생성 결과를 육안으로 점검하는 실험이며, 분류기는 학습하지 않습니다. 동등한 성능이나 실제 결함 재현은 이 실험만으로 증명되지 않습니다. 결과와 재현 정보는 Drive의 새 폴더에 저장하고, 종료·오류 시 G4 반납을 시도합니다. 연결이 강제로 끊어지면 자동 반납을 보장할 수 없습니다.

In [ ]:
from google.colab import drive, runtime
import hashlib, json, os, subprocess, sys, time, traceback, zipfile
from pathlib import Path, PurePosixPath

mounted = False
run_root = None
created_run = False
try:
    drive.mount('/content/drive')
    mounted = True
    import torch
    from PIL import Image, ImageOps, ImageDraw

    drive_root = Path('/content/drive/MyDrive/산공종설')
    data_zip = drive_root / 'gan_clean_underfill_50_vs_300_data.zip'
    code_zip = drive_root / 'gan_underfill_50_vs_300_code.zip'
    run_root = drive_root / 'results/gan_clean_underfill_50_vs_300_20261101_run01'
    local = Path('/content/gan_underfill_50_vs_300')
    prepared = local / 'prepared'
    model_path = local / 'gan_model_count_generalized.py'
    seed, steps, batch = 42, 1500, 8
    def sha(path):
        h = hashlib.sha256()
        with path.open('rb') as f:
            for b in iter(lambda: f.read(1024*1024), b''): h.update(b)
        return h.hexdigest()
    for p in (data_zip, code_zip):
        if not p.is_file(): raise FileNotFoundError(f'필요한 Drive 파일 없음: {p}')
    if sha(data_zip) != 'd4fe5b65e1de9dbff3604812bb3677027712aa1ff20b136e0a0c1277a937ffee' or sha(code_zip) != 'c252b75d1d89b9a0f5a7901f3543351cc7bc1d4e988cb609095f1f69e049114a':
        raise ValueError('입력 묶음 또는 코드 묶음의 해시가 다릅니다.')
    if not torch.cuda.is_available(): raise RuntimeError('GPU가 없습니다. G4를 선택하세요.')
    gpu = torch.cuda.get_device_name(0)
    print('사용 GPU:', gpu, flush=True)
    if 'RTX PRO 6000' not in gpu.upper(): raise RuntimeError(f'G4가 아닙니다: {gpu}')
    if run_root.exists():
        raise FileExistsError(f'기존 결과를 보호하기 위해 새 이름을 사용하세요: {run_root}')
    run_root.mkdir(parents=True); created_run = True
    for d in ('logs', 'review', 'gan', 'input'): (run_root/d).mkdir()
    local.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(code_zip) as z:
        if z.namelist() != ['gan_model_count_generalized.py']: raise ValueError('코드 묶음 구성 불일치')
        model_path.write_bytes(z.read(z.namelist()[0]))
    if sha(model_path) != '0d85d1ba9580e46086e1370e8c72eaa68a6b08e5f94f0243a98497560960767d': raise ValueError('모델 파일 해시 불일치')
    prepared.mkdir(exist_ok=True)
    with zipfile.ZipFile(data_zip) as z:
        if z.testzip() is not None: raise ValueError('입력 ZIP 손상')
        for info in z.infolist():
            parts = PurePosixPath(info.filename).parts
            if not parts or info.filename.startswith('/') or any(x in ('..','') for x in parts) or '\\' in info.filename:
                raise ValueError(f'잘못된 경로: {info.filename}')
            dest = prepared.joinpath(*parts)
            if info.is_dir(): dest.mkdir(parents=True, exist_ok=True)
            else:
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(z.read(info))
    sets = {}
    for label, count in [('underfill_50',50),('underfill_300',300)]:
        paths = sorted((prepared/'data'/label).glob('*.png'))
        if len(paths) != count: raise ValueError(f'{label}: 실제 {len(paths)}장, {count}장 필요')
        for p in paths:
            with Image.open(p) as im:
                if im.mode != 'RGB' or im.size != (256,256):
                    raise ValueError(f'RGB 256×256 입력 아님: {p}')
        sets[label] = {p.name: sha(p) for p in paths}
    if not all(sets['underfill_300'].get(k) == v for k,v in sets['underfill_50'].items()):
        raise ValueError('50장 목록이 300장 목록에 동일한 픽셀 파일로 포함되지 않습니다.')
    for name in ('input_manifest.csv','selection_settings.json'):
        src = prepared/name
        if not src.exists(): raise FileNotFoundError(f'입력 기록 누락: {src}')
        (run_root/'input'/name).write_bytes(src.read_bytes())
    settings = {'conditions': {'underfill_50':50,'underfill_300':300}, 'seed':seed,
                'steps':steps,'batch':batch,'generated_per_condition':50,'gpu':gpu,
                'original_gan_sha256':'8e10880d3c8fa3c897d0775b745f60765ec91a3dc01bed304e168a307797c961','modified_model_sha256':'0d85d1ba9580e46086e1370e8c72eaa68a6b08e5f94f0243a98497560960767d',
                'code_zip_sha256':'c252b75d1d89b9a0f5a7901f3543351cc7bc1d4e988cb609095f1f69e049114a','data_zip_sha256':'d4fe5b65e1de9dbff3604812bb3677027712aa1ff20b136e0a0c1277a937ffee',
                'source_model_change':'50장 정확히 강제하던 입력 가드를 비어있는 입력 금지로 변경. 장수·포함관계는 노트북에서 검증.',
                'expected_sample_draws_per_image':{'50':240,'300':40},
                'comparison_limit':'복원 무작위 추출, 동일 업데이트 예산이라 사진 1장당 선택 횟수가 다름',
                'warning':'표면 선별은 자동 추정이며 완전성 미보장. 출력물마다 출력 1회. 분류기 평가 미수행.'}
    (run_root/'settings.json').write_text(json.dumps(settings,ensure_ascii=False,indent=2),encoding='utf-8')

    def run_logged(args, path):
        print('실행:', ' '.join(map(str,args)),flush=True)
        with path.open('w',encoding='utf-8') as f:
            proc = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
            try:
                for line in proc.stdout:
                    f.write(line);f.flush();print(line,end='',flush=True)
                status = proc.wait()
            except BaseException:
                proc.terminate(); raise
            if status: raise RuntimeError(f'학습 오류 ({status}): {path}')

    def sheet(paths, dest):
        tile, cap, cols = 180, 24, 10
        board = Image.new('RGB',(cols*tile,((len(paths)+cols-1)//cols)*(tile+cap)), 'white')
        draw = ImageDraw.Draw(board)
        for i,p in enumerate(paths):
            x,y = i%cols*tile, i//cols*(tile+cap)
            with Image.open(p) as im:
                board.paste(ImageOps.fit(im.convert('RGB'),(tile,tile),method=Image.Resampling.LANCZOS),(x,y))
            draw.text((x+3,y+tile+3),p.stem[:23],fill='black')
        board.save(dest,quality=93)
    summary = {}
    for i,(label,count) in enumerate([('underfill_50',50),('underfill_300',300)],1):
        actual = sorted((prepared/'data'/label).glob('*.png'))
        gan = run_root/'gan'/label
        print(f'[{i}/2] 실제 {count}장으로 학습 시작',flush=True)
        args = [sys.executable,model_path,'--data',prepared/'data'/label,'--out',gan,
                '--steps',steps,'--seed',seed,'--num-generate',50,'--device','cuda',
                '--batch',batch,'--save-every',100,'--log-every',25]
        run_logged(args,run_root/'logs'/f'{label}.log')
        progress = json.loads((gan/'progress.json').read_text(encoding='utf-8'))
        manifest = json.loads((gan/'training_manifest.json').read_text(encoding='utf-8'))
        generated = sorted((gan/'generated').glob('*.png'))
        if progress.get('status') != 'completed' or progress.get('step') != steps:
            raise RuntimeError(f'{label} 완료 상태 불일치: {progress}')
        if [p.name for p in generated] != [f'generated_{j:03d}.png' for j in range(1,51)]:
            raise RuntimeError(f'{label} 생성 50장 확인 실패')
        if {r['filename']:r['sha256'] for r in manifest['images']} != sets[label]:
            raise RuntimeError(f'{label}: 실제 학습 이미지 목록 불일치')
        sheet(generated,run_root/'review'/f'{label}_generated_50.jpg')
        sheet(actual,run_root/'review'/f'{label}_actual_{count}.jpg')
        summary[label] = {'training_count':count, 'generated':len(generated),'status':'completed'}
        print(f'[{i}/2] 실제 {count}장 조건 생성 완료',flush=True)
    (run_root/'README_실험설명.md').write_text("""# 실제 언더필 50장 대 300장 GAN 예비 비교

각 모델은 DCGAN을 동일한 seed 42, batch 8, 1500 업데이트로 처음부터 학습하며 50장씩 생성한다. 50장 사진은 300장에 포함된다. 동일 업데이트 수의 무작위 복원 추출이므로 사진당 평균 추출 횟수는 각각 약 240회와 40회이며, 품질 차이를 이미지 수 한 요소 때문이라고 단정할 수 없다.

코드는 기존 gan_model.py에서 입력 사진 50장 강제만 제거한 복사본이다. 입력 장수와 포함 관계는 노트북에서 확인하고, 원본/복사본과 묶음의 SHA256은 settings.json에 저장한다.

입력 선별·프레임·분할 기준은 input 폴더의 input_manifest.csv와 selection_settings.json을 확인한다. 마스크는 바깥 영역 선별 기준으로만 사용하고 실제 학습 입력에는 칠하지 않는다. 자동 선별은 베드가 없거나 결함이 선명하다는 것을 보장하지 않는다.

현재 실험은 생성 예시를 보는 용도이며 분류기 학습 또는 성능 검증은 하지 않는다. 원본은 한 클래스당 출력 1회이므로 독립 출력에 대한 일반화는 확인할 수 없다. 체크포인트는 Drive gan 폴더에 보관하고 결과 ZIP에서는 뺐다.
""",encoding='utf-8')
    result = run_root/'gan_clean_underfill_50_vs_300_results.zip'
    tmp = result.with_suffix('.zip.part')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=4) as z:
        for root in (run_root/'input',run_root/'gan',run_root/'review',run_root/'logs'):
            for p in root.rglob('*'):
                if not p.is_file() or p.suffix.lower() in ('.pt','.pth','.tmp') or 'previews' in p.relative_to(run_root).parts: continue
                z.write(p,p.relative_to(run_root).as_posix())
        for p in (run_root/'README_실험설명.md',run_root/'settings.json'):
            z.write(p,p.relative_to(run_root).as_posix())
    with zipfile.ZipFile(tmp) as z:
        if z.testzip() is not None: raise RuntimeError('결과 ZIP 무결성 오류')
    os.replace(tmp,result)
    (run_root/'run_status.json').write_text(json.dumps({'status':'completed','summary':summary,
        'result_zip':str(result),'classifier_training':False},ensure_ascii=False,indent=2),encoding='utf-8')
    print('두 조건 생성 완료. 결과 ZIP:',result,flush=True)
except BaseException as exc:
    trace = traceback.format_exc(); print('오류:',trace,flush=True)
    if created_run and run_root is not None:
        try: (run_root/'run_status.json').write_text(json.dumps({'status':'failed','error':str(exc),'traceback':trace},ensure_ascii=False,indent=2),encoding='utf-8')
        except Exception as save_exc: print('오류 기록 실패:',save_exc,flush=True)
finally:
    if mounted:
        try: print('Drive 동기화 중',flush=True); drive.flush_and_unmount()
        except BaseException as exc: print('Drive 동기화 확인 필요:',exc,flush=True)
    print('G4 런타임 반납 요청',flush=True)
    try: runtime.unassign()
    except BaseException as exc: print('자동 반납 실패:',exc,flush=True)


## 결과

Drive의 `산공종설/results/gan_clean_underfill_50_vs_300_20261101_run01/`에서 입력 기록, 로그, 조건별 학습·생성 사진 확인판, 체크포인트 및 결과 ZIP을 확인합니다.